### 检验manifest中dev的数据

In [1]:
import pandas as pd

manifest = pd.read_csv(
    r"D:\Projects\speech-ai-learning\metadata\manifest.csv"
)

dev = manifest[manifest["split"] == "dev"].copy()

print("Dev 样本数量：", len(dev))
print("字段：", dev.columns.tolist())
print(dev.head())

Dev 样本数量： 409
字段： ['audio_path', 'speaker_id', 'text', 'source_id', 'parent_uid', 'start_time', 'end_time', 'split']
                           audio_path  \
2281  common_voice_zh-CN_18600769.mp3   
2282  common_voice_zh-CN_18600881.mp3   
2283  common_voice_zh-CN_18600767.mp3   
2284  common_voice_zh-CN_18600768.mp3   
2285  common_voice_zh-CN_18641893.mp3   

                                             speaker_id  \
2281  5d66e9ec552b7b74ae9c1137559ea3cdfe341af1e3f898...   
2282  5d66e9ec552b7b74ae9c1137559ea3cdfe341af1e3f898...   
2283  5d66e9ec552b7b74ae9c1137559ea3cdfe341af1e3f898...   
2284  5d66e9ec552b7b74ae9c1137559ea3cdfe341af1e3f898...   
2285  5d66e9ec552b7b74ae9c1137559ea3cdfe341af1e3f898...   

                          text                     source_id  \
2281  尤因镇区是位于美国阿肯色州布恩县的一个行政镇区。  common_voice_beijing26_zh-CN   
2282  莫尔特里是一个位于美国乔治亚州科尔奎特县的城市。  common_voice_beijing26_zh-CN   
2283                 无法联系或拒绝电访  common_voice_beijing26_zh-CN   
2284               一级造园施工

### 创建dev-candidates 409条候选队列

In [2]:
#创建候选队列
from pathlib import Path
import pandas as pd
import hashlib

# 路径
root = Path(r"D:\Projects\speech-ai-learning")

manifest_path = root / "metadata" / "manifest.csv"
clips_dir = (
    root / "data" / "raw" /
    "common_voice_beijing26" /
    "zh-CN-beijing" / "clips"
)

output_path = (
    root / "metadata" / "dev_candidates_W7D5.csv"
)

#防止覆盖已有文件
assert not output_path.exists(), (
    f"候选文件已经存在：{output_path}"
)

#读取Dev
manifest = pd.read_csv(manifest_path)
dev = manifest[
    manifest["split"] == "dev"
].copy()

assert len(dev) == 409

#检查必要字段
required_columns = [
    "audio_path",
    "speaker_id",
    "text",
    "source_id",
    "parent_uid",
    "start_time",
    "end_time",
    "split"
]

assert all(
    col in dev.columns
    for col in required_columns
)

assert dev["parent_uid"].notna().all()
assert dev["audio_path"].notna().all()

# 检查 UID 是否重复
assert dev["parent_uid"].is_unique

# =====================
# 3. 固定排序后随机打乱
# =====================

# 先按照 parent_uid 排序，
# 确保随机打乱前的输入顺序固定

dev = dev.sort_values(
    "parent_uid"
).reset_index(drop=True)

# 全部409条随机打乱一次
# 固定随机种子，保证可复现

dev_candidates = dev.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

#分配固定selection_rank
dev_candidates["selection_rank"] = range(
    1,
    len(dev_candidates) + 1
)


# =====================
# 4. 构建候选记录
# =====================

records = []

for _, row in dev_candidates.iterrows():

    #音频文件名
    audio_name = Path(
        row["audio_path"]
    ).name

    # 完整音频路径
    audio_path = clips_dir / audio_name

    # 检查音频是否存在
    assert audio_path.exists(), (
        f"音频不存在：{audio_path}"
    )

    # 根据原始 audio_path 生成稳定 UID
    uid_hash = hashlib.sha256(
        str(row["audio_path"]).encode("utf-8")
    ).hexdigest()[:16]

    candidate_uid = f"cv26_{uid_hash}"

    # 音频时长
    duration_sec = (
        row["end_time"] - row["start_time"]
    )

    records.append({

        "selection_rank": (
            len(records) + 1
        ),

        "candidate_uid": candidate_uid,

        "audio_path": str(audio_path),

        "duration_sec": duration_sec,

        "speaker_id": row["speaker_id"],

        "parent_uid": row["parent_uid"],

        "source_text": row["text"],

        "raw_text": "",

        "score_text": "",

        "status": "pending",

        "review_minutes": "",

        "split": "dev"
    })

# =====================
# 5. 转换为 DataFrame
# =====================

dev_candidates = pd.DataFrame(records)

# =====================
# 6. 完整性检查
# =====================

assert len(dev_candidates) == 409

assert dev_candidates["candidate_uid"].is_unique

assert dev_candidates["parent_uid"].is_unique

assert (
    dev_candidates["status"] == "pending"
).all()

assert (
    dev_candidates["split"] == "dev"
).all()

assert dev_candidates["audio_path"].apply(
    lambda x: Path(x).exists()
).all()


# =====================
# 7. 保存候选队列
# =====================

dev_candidates.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

# 8. 输出检查结果
# =====================

print("W7D5 Dev 候选队列创建成功")

print("候选数量:", len(dev_candidates))

print("\n状态统计:")
print(
    dev_candidates["status"].value_counts()
)

print("\n字段检查:")
for col in [
    "raw_text",
    "score_text",
    "review_minutes"
]:
    print(
        f"{col} 为空:",
        dev_candidates[col].isna().sum()
        + (dev_candidates[col] == "").sum()
    )

print("\n前10条候选:")
print(
    dev_candidates[
        [
            "selection_rank",
            "candidate_uid",
            "audio_path",
            "duration_sec",
            "status"
        ]
    ].head(10).to_string(index=False)
)

print("\n已保存:", output_path)

print("\nDev 候选队列完整性检查通过")

W7D5 Dev 候选队列创建成功
候选数量: 409

状态统计:
status
pending    409
Name: count, dtype: int64

字段检查:
raw_text 为空: 409
score_text 为空: 409
review_minutes 为空: 409

前10条候选:
 selection_rank         candidate_uid                                                                                                         audio_path  duration_sec  status
              1 cv26_3f66c228bc2a4ae8 D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_22057623.mp3      5.957625 pending
              2 cv26_1753be3b6083f28f D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_22004281.mp3      2.789625 pending
              3 cv26_20558b6a5c8dbac9 D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_22004016.mp3      5.621625 pending
              4 cv26_40123f9681ffefa4 D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_18

### 创建音频展示函数和标注更新函数

In [3]:
#读取音频并展示
from pathlib import Path
import pandas as pd
import soundfile as sf
from IPython.display import Audio, display

root = Path.cwd().resolve()
if root.name.lower() == "notebooks":
    root = root.parent

candidate_path = root / "metadata" / "dev_candidates_W7D5.csv"

audio_root = (
    root / "data" / "raw"
    / "common_voice_beijing26"
    / "zh-CN-beijing"
)

def resolve_audio_path(audio_path):
    p = Path(str(audio_path))

    candidates = (
        [p] if p.is_absolute() else [
            root / p,
            audio_root / p,
            audio_root / "clips" / p
        ]
    )

    for candidate in candidates:
        if candidate.is_file():
            return candidate

    raise FileNotFoundError(
        f"找不到音频:{audio_path}"
    )

def show_candidate(rank):
    df = pd.read_csv(
        candidate_path,
        keep_default_na=False
    )

    selected = df.loc[
        df["selection_rank"] == rank
    ]

    if len(selected) != 1:
        raise ValueError("没有找到唯一候选样本")

    row = selected.iloc[0]

    #if row["review_batch"] != "W5D4" and row["review_batch"] != "W5D5":
        #raise ValueError("该样本不属于W5D4/W5D5")

    path = resolve_audio_path(row["audio_path"])

    #读取音频
    audio, sr = sf.read(
        path,
        dtype="float32",
        always_2d=True
    )

    #如果有多个声道，简单取平均
    waveform = audio.mean(axis=1)

    print("序号:", rank)
    print("UID:", row["candidate_uid"])
    print("音频时长:", len(waveform) / sr)
    print("当前状态:", row["status"])
    print("原始转写:", row["source_text"])

    display(Audio(waveform, rate=sr))

In [4]:
#标注更新函数
def update_candidate(
        rank,
        raw_text,
        status,
        score_text="",
        uncertain_span="",
        notes="",
        review_minutes=""
):
    if status not in ["verified","needs_review"]:
        raise ValueError("无效标注状态")

    if status == "verified":
        if not str(raw_text).strip():
            raise ValueError("verified必须有确认后的文本")

        if str(uncertain_span).strip():
            raise ValueError(
                "仍存在不确定的片段，不能标记verified"
            )

    if status == "needs_review":
        if not str(uncertain_span).strip() and not str(notes).strip():
            raise ValueError("请记录需要复核的原因")

    df = pd.read_csv(
        candidate_path,
        dtype=str,
        keep_default_na=False
    )

    mask = df["selection_rank"] == str(rank)

    if mask.sum() != 1:
        raise ValueError("没有找到唯一候选样本")


    df.loc[mask, "raw_text"] = raw_text
    df.loc[mask, "score_text"] = score_text
    df.loc[mask, "status"] = status
    df.loc[mask, "uncertain_span"] = uncertain_span
    df.loc[mask, "notes"] = notes
    df.loc[mask, "review_minutes"] = str(review_minutes)

    #临时文件写入成功后，再替换原CSV
    temp_path = candidate_path.with_suffix(".csv.tmp")

    df.to_csv(
        temp_path,
        index=False,
        encoding="utf-8-sig"
    )

    temp_path.replace(candidate_path)

    print(f"第{rank}条已更新为：{status}")

### 2.更新1-10条候选队列

In [5]:
#检验队列
import pandas as pd

path = (
    r"D:\Projects\speech-ai-learning"
    r"\metadata\dev_candidates_W7D5.csv"
)

df = pd.read_csv(path)

batch = (
    df[df["status"] == "pending"]
    .head(10)
)

print(batch[
    [
        "selection_rank",
        "candidate_uid",
        "audio_path",
        "duration_sec",
        "source_text",
        "status"
    ]
].to_string(index=False))

assert len(batch) == 10

 selection_rank         candidate_uid                                                                                                         audio_path  duration_sec                           source_text  status
              1 cv26_3f66c228bc2a4ae8 D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_22057623.mp3      5.957625                       这两家公司都在网上出售树莓派。 pending
              2 cv26_1753be3b6083f28f D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_22004281.mp3      2.789625                                武藏国出身。 pending
              3 cv26_20558b6a5c8dbac9 D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_22004016.mp3      5.621625                     另一联赛是阿根廷全国足球甲级联赛。 pending
              4 cv26_40123f9681ffefa4 D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_18994260

In [6]:
#展示10条音频
for i in range(1,11):
    show_candidate(i)

序号: 1
UID: cv26_3f66c228bc2a4ae8
音频时长: 5.952
当前状态: pending
原始转写: 这两家公司都在网上出售树莓派。


序号: 2
UID: cv26_1753be3b6083f28f
音频时长: 2.784
当前状态: pending
原始转写: 武藏国出身。


序号: 3
UID: cv26_20558b6a5c8dbac9
音频时长: 5.616
当前状态: pending
原始转写: 另一联赛是阿根廷全国足球甲级联赛。


序号: 4
UID: cv26_40123f9681ffefa4
音频时长: 6.384
当前状态: pending
原始转写: 措布伦是奥地利蒂罗尔州罗伊特县的一个市镇。


序号: 5
UID: cv26_8e8d4f65babf0b7e
音频时长: 8.976
当前状态: pending
原始转写: 维济马农村居民点是俄罗斯联邦沃洛格达州别洛泽尔斯克区所属的一个农村居民点。


序号: 6
UID: cv26_95b35d7219a5eeb9
音频时长: 5.016
当前状态: pending
原始转写: 也是日本电视控股事业的子公司之一。


序号: 7
UID: cv26_dc3e083eab508fd5
音频时长: 4.392
当前状态: pending
原始转写: 编组共两节车厢。


序号: 8
UID: cv26_f05564f22bc1d77c
音频时长: 6.576
当前状态: pending
原始转写: 德国空军将西班牙内战学习的经验第一次运用在大规模战争上。


序号: 9
UID: cv26_2bd1b28412804c60
音频时长: 7.956
当前状态: pending
原始转写: 在接下来的几个月里，“信天翁”在旧金山境外运营。


序号: 10
UID: cv26_1441a8cec768fc20
音频时长: 8.496
当前状态: pending
原始转写: 阿韦利诺洛佩斯是巴西皮奥伊州的一个市镇。


In [10]:
#标注更新
from pathlib import Path
import sys

root = Path(r"D:\Projects\speech-ai-learning")

if root not in sys.path:
    sys.path.insert(0, str(root))

from src.normalize import normalize_score

update_candidate(
    1,
    raw_text="这两家公司都在网上出售树莓派。",
    status="verified",
    score_text=normalize_score("这两家公司都在网上出售树莓派。"),
    review_minutes="0.5"
)

update_candidate(
    2,
    raw_text="武藏国出身。",
    status="verified",
    score_text=normalize_score("武藏国出身。"),
    review_minutes="0.3"
)

update_candidate(
    3,
    raw_text="另一联赛是阿根廷全国足球甲级联赛。",
    status="verified",
    score_text=normalize_score("另一联赛是阿根廷全国足球甲级联赛。"),
    review_minutes="0.4"
)

update_candidate(
    4,
    raw_text="措布伦是奥地利蒂罗尔州罗伊特县的一个市镇。",
    status="verified",
    score_text=normalize_score("措布伦是奥地利蒂罗尔州罗伊特县的一个市镇。"),
    review_minutes="0.5"
)

update_candidate(
    5,
    raw_text="维济马农村居民点是俄罗斯联邦沃洛格达州别洛泽尔斯克区所属的一个农村居民点。",
    status="verified",
    score_text=normalize_score("维济马农村居民点是俄罗斯联邦沃洛格达州别洛泽尔斯克区所属的一个农村居民点。"),
    review_minutes="0.8"
)

update_candidate(
    6,
    raw_text="也是日本电视控股事业的[uncertain]公司之一。",
    status="needs_review",
    score_text=normalize_score(""),
    review_minutes="1.0",
    uncertain_span="事业的与公司之间存在一个音节，但无法辨认是否为子",
    notes="录音质量问题，反复回听仍无法确认具体字词，暂不通过验收。"
)

update_candidate(
    7,
    raw_text="编组共两节车厢。",
    status="verified",
    score_text=normalize_score("编组共两节车厢。"),
    review_minutes="0.4"
)

update_candidate(
    8,
    raw_text="德国空军将西班牙内战学习的经验第一次运用在大规模战争上。",
    status="verified",
    score_text=normalize_score("德国空军将西班牙内战学习的经验第一次运用在大规模战争上。"),
    review_minutes="0.5"
)

update_candidate(
    9,
    raw_text="在接下来的几个月里，“信天翁”在旧金山境外运营。",
    status="verified",
    score_text=normalize_score("在接下来的几个月里，“信天翁”在旧金山境外运营。"),
    review_minutes="0.5"
)

update_candidate(
    10,
    raw_text="阿韦利诺洛佩斯是巴西皮奥伊州的一个市镇。",
    status="verified",
    score_text=normalize_score("阿韦利诺洛佩斯是巴西皮奥伊州的一个市镇。"),
    review_minutes="0.5"
)

第1条已更新为：verified
第2条已更新为：verified
第3条已更新为：verified
第4条已更新为：verified
第5条已更新为：verified
第6条已更新为：needs_review
第7条已更新为：verified
第8条已更新为：verified
第9条已更新为：verified
第10条已更新为：verified


In [12]:
#由于第6条无法辨认，所以加一条11
show_candidate(11)
update_candidate(
    11,
    raw_text="母孙氏。",
    status="verified",
    score_text=normalize_score("母孙氏。"),
    review_minutes="0.4"
)

序号: 11
UID: cv26_dc1278883fe4eb06
音频时长: 2.784
当前状态: pending
原始转写: 母孙氏。


第11条已更新为：verified


In [13]:
#检查本批次状态
import pandas as pd

path = (
    r"D:\Projects\speech-ai-learning"
    r"\metadata\dev_candidates_W7D5.csv"
)

df = pd.read_csv(path)

print("状态统计：")
print(df["status"].value_counts())

print("\n字段检查：")
for col in [
    "raw_text",
    "score_text",
    "review_minutes"
]:
    print(
        f"{col} 为空：",
        df.loc[
            df["status"] == "verified", col
        ].isna().sum()
    )

print(
    "\n累计 Dev verified:",
    (df["status"] == "verified").sum()
)

assert (
    df["status"] == "verified"
).sum() == 10

verified_df = df[df["status"] == "verified"]

assert verified_df[
    ["raw_text", "score_text", "review_minutes"]
].notna().all().all()

assert (
    verified_df["raw_text"].astype(str).str.strip() != ""
).all()

assert (
    verified_df["score_text"].astype(str).str.strip() != ""
).all()

print("\nW7D5 Dev 标注验收通过")

状态统计：
status
pending         398
verified         10
needs_review      1
Name: count, dtype: int64

字段检查：
raw_text 为空： 0
score_text 为空： 0
review_minutes 为空： 0

累计 Dev verified: 10

W7D5 Dev 标注验收通过
